# 02 · Train/test split
Applies the column decisions from notebook 01 and creates the stratified 80/20 split used by every later notebook. After this notebook, the test set stays untouched until the final evaluation in notebook 06.

## 1. Load the data
**What it does:** loads `data.csv` and removes the leading spaces from the column names.

**Why:** the raw file has a space at the start of every column name, which makes columns hard to reference.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/data.csv")
df.columns = df.columns.str.strip()

## 2. Drop the constant and duplicate columns
**What it does:** removes three columns: `Net Income Flag`, which has the same value for every record, and one column from each of the two exact-duplicate pairs found in notebook 01.

**Why:** these columns carry no extra information. Dropping them is based on the structure of the columns (constant or identical copies), not on statistics learned from the data, so it is safe to do before the split.

In [ ]:
cols_to_drop = [
    "Net Income Flag",                  # same value for every company
    "Current Liability to Liability",   # exact copy of "Current Liabilities/Liability"
    "Current Liability to Equity",      # exact copy of "Current Liabilities/Equity"
]
df = df.drop(columns=cols_to_drop)

## 3. Separate the inputs (X) from the target (y)
**What it does:** puts the 92 financial ratios in `X` and the `Bankrupt?` column in `y`.

**Why:** models learn to predict `y` from `X`, so the two must be kept apart.

In [ ]:
X = df.drop(columns="Bankrupt?")
y = df["Bankrupt?"]

## 4. Stratified 80/20 train/test split
**What it does:** shuffles the records and splits them into 80% training and 20% test, keeping the same bankruptcy rate in both (`stratify=y`), with a fixed seed (`random_state=42`).

**Why:** the test set plays the role of new, unseen records and is only opened in notebook 06. Stratifying matters because only 3.2% of records are bankrupt; the seed makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% kept aside for the final test
    stratify=y,         # keep the same bankruptcy rate in both sets
    random_state=42,    # same split every time
)

## 5. Check the split
**What it does:** prints the number of features and, for each set, the number of records, bankrupt records and bankruptcy rate.

**Why:** confirms stratification worked: both sets should show about 3.23% bankrupt.

In [ ]:
print("Features:", X.shape[1])
print("Training set:", X_train.shape[0], "companies |", y_train.sum(), "bankrupt |",
      round(y_train.mean() * 100, 2), "%")
print("Test set:    ", X_test.shape[0], "companies |", y_test.sum(), "bankrupt |",
      round(y_test.mean() * 100, 2), "%")

Features: 92
Training set: 5455 companies | 176 bankrupt | 3.23 %
Test set:     1364 companies | 44 bankrupt | 3.23 %


## 6. Save both sets
**What it does:** writes the training and test sets, each with its `Bankrupt?` column, to `data/train.csv` and `data/test.csv`.

**Why:** every later notebook loads these files, so all of them use exactly the same split and the test set is never re-created differently.

In [ ]:
X_train.assign(**{"Bankrupt?": y_train}).to_csv("data/train.csv", index=False)
X_test.assign(**{"Bankrupt?": y_test}).to_csv("data/test.csv", index=False)
print("Saved data/train.csv and data/test.csv")

Saved data/train.csv and data/test.csv
